In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Load all JSON files from downloaded folder
data_dir = Path('../data/downloaded')
all_records = []

print("=== LOADING DATA ===")
for json_file in data_dir.rglob('*.json'):
    try:
        with open(json_file, 'r') as f:
            file_data = json.load(f)
            if isinstance(file_data, dict) and 'records' in file_data:
                all_records.extend(file_data['records'])
            elif isinstance(file_data, list):
                all_records.extend(file_data)
    except Exception as e:
        print(f"Error reading {json_file}: {e}")

print(f"Loaded {len(all_records)} records")

# Create DataFrame
df = pd.DataFrame(all_records)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Sort by city and time - CRITICAL for time-series features!
df = df.sort_values(['city', 'timestamp']).reset_index(drop=True)

# Verify city column exists
print(f"\n✓ 'city' column exists: {'city' in df.columns}")
print(f"Cities: {df['city'].unique().tolist()}")

print("\n=== CREATING FEATURES ===")

# 1. Time-based features
print("Creating time-based features...")
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['day_of_month'] = df['timestamp'].dt.day
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# 2. Lag features - using transform instead of apply
print("Creating lag features...")
df['temp_lag_1'] = df.groupby('city')['temperature_celsius'].shift(1)
df['temp_lag_2'] = df.groupby('city')['temperature_celsius'].shift(2)
df['temp_lag_4'] = df.groupby('city')['temperature_celsius'].shift(4)
df['humidity_lag_1'] = df.groupby('city')['humidity_percent'].shift(1)
df['pressure_lag_1'] = df.groupby('city')['pressure_hpa'].shift(1)

# 3. Rolling statistics - using transform
print("Creating rolling statistics...")
df['temp_rolling_mean_24h'] = df.groupby('city')['temperature_celsius'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
df['temp_rolling_std_24h'] = df.groupby('city')['temperature_celsius'].transform(lambda x: x.rolling(window=4, min_periods=1).std())
df['temp_rolling_max_24h'] = df.groupby('city')['temperature_celsius'].transform(lambda x: x.rolling(window=4, min_periods=1).max())
df['temp_rolling_min_24h'] = df.groupby('city')['temperature_celsius'].transform(lambda x: x.rolling(window=4, min_periods=1).min())

# 4. Temperature change
print("Creating temperature change features...")
df['temp_change_6h'] = df.groupby('city')['temperature_celsius'].diff()

# 5. Target variable
print("Creating target variable...")
df['target_temp_24h'] = df.groupby('city')['temperature_celsius'].shift(-4)

# 6. One-hot encode
print("One-hot encoding city...")
df = pd.get_dummies(df, columns=['city'], prefix='city')

print(f"\n✓ Total features: {len(df.columns)}")

# Remove NaN rows
df_clean = df.dropna()
print(f"✓ Records after removing NaN: {len(df_clean)}")

# Save
df_clean.to_csv('../data/processed/featured_data.csv', index=False)
print("\nFeature engineering complete")

# Show sample
print("\n=== SAMPLE OF FEATURED DATA ===")
print(df_clean.head())
print("\n=== FEATURES CREATED ===")
print(df_clean.columns.tolist())

In [ ]:
# Check the first raw record before converting to DataFrame
print("First record from JSON:")
print(all_records[0])
print("\nKeys in first record:")
print(all_records[0].keys())

In [ ]:
# Quick feature importance check
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Load featured data
df = pd.read_csv('../data/processed/featured_data.csv')

# Separate features and target
feature_cols = [col for col in df.columns if col not in ['timestamp', 'target_temp_24h', 
                                                           'weather_description', 'country',
                                                           'latitude', 'longitude']]
X = df[feature_cols]
y = df['target_temp_24h']

# Quick random forest to see feature importance
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

# Plot feature importance
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Top 15 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=300)
plt.show()

print("Feature importance analysis complete")